In [0]:
import pandas as pd


In [0]:
%pip install openpyxl

In [0]:
df = pd.read_excel("/Volumes/uci_retail/bronze/online_retail/Online Retail.xlsx")

In [0]:
df.head(10)

In [0]:
df.shape


In [0]:
df.columns

In [0]:
df.dtypes


In [0]:
df.isnull().sum()


In [0]:
spark_df = spark.createDataFrame(df.astype({col: str for col in df.select_dtypes(include=['object']).columns}))

In [0]:
spark_df.printSchema()

In [0]:
spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("uci_retail.bronze.online_retail")

In [0]:
spark.table("uci_retail.bronze.online_retail").count()

In [0]:
display(
    spark.table("uci_retail.bronze.online_retail").limit(10)
)

In [0]:
silver_df = spark.table("uci_retail.bronze.online_retail")

In [0]:
display(silver_df)

In [0]:
display(
    silver_df
    .filter(silver_df.InvoiceNo.startswith("C"))
    .limit(20)
)

In [0]:
silver_df.filter(
    silver_df.InvoiceNo.startswith("C")
).count()

In [0]:
from pyspark.sql.functions import when, col

In [0]:
silver_df = silver_df.withColumn(
    "TransactionType",
    when(col("InvoiceNo").startswith("C"), "Cancellation")
    .otherwise("Sale")
)

In [0]:
display(
    silver_df.groupBy("TransactionType").count()
)

In [0]:
# ============================================================
# METADADOS — TransactionType
# ============================================================
# Nome: TransactionType
# Tipo: STRING
# Origem: Coluna InvoiceNo
# Camada: Silver
#
# Descrição:
# Classifica cada registro de acordo com o tipo de transação.
#
# Regra de negócio:
# - InvoiceNo iniciado por "C" = Cancellation
# - Qualquer outro InvoiceNo = Sale
#
# Objetivo:
# Permitir que análises diferenciem vendas de cancelamentos
# sem excluir os registros de cancelamento da camada Silver.
#
# Importante:
# Esta coluna é derivada dos dados originais e não substitui
# o valor original de InvoiceNo.
# ============================================================

silver_df = silver_df.withColumn(
    "TransactionType",
    when(col("InvoiceNo").startswith("C"), "Cancellation")
    .otherwise("Sale")
)

In [0]:
# Pipeline de Engenharia de Dados - UCI Online Retail